<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP10.1A/MP10.1A_dfn_lfp.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP10.1A · The Full Model, Where the SPMe Stops

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# --- PyBaMM ------------------------------------------------------------------
# The Doyle-Fuller-Newman model is solved with PyBaMM (pybamm.org), which Colab does not have.
# This cell installs it once per session (about a minute); locally it does nothing if PyBaMM is there.
import importlib.util, subprocess, sys
if importlib.util.find_spec("pybamm") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybamm==26.10.1.1"], check=True)
import pybamm
print("PyBaMM", pybamm.__version__)

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/")
PROJECT = (['mp101a.py', 'MP10.1A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP10.1A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex10.1-spm-and-thermal", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex10.1-spm-and-thermal", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp101a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp101a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP10.1A_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex10.1's problem.py, by its name in Ex10.1

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

The capacity printed on a cell's datasheet is what it delivers at a low current. At a high current it delivers less, and its voltage sags: lithium cannot reach the reaction sites fast enough, through the particles and through the electrolyte in the pores. An engineer sizing a battery pack for an electric bus or a grid-support unit, or writing the state-of-charge estimator of a battery management system, needs **the voltage of a discharge and the capacity delivered, at any current the application asks for**.

The model that predicts both from the cell's physics is the **Doyle-Fuller-Newman model** (DFN, also called the pseudo-two-dimensional model): lithium in the electrolyte across the cell, lithium in a particle at every place in each electrode, the potentials of the solid and of the electrolyte, and the reaction between them. It is the reference of battery modelling, and it costs a full solve for every new current. Simpler models - the single particle model of Ex10.1, and its extension with the electrolyte, the **SPMe** - are fast but hold only while the electrolyte is not limiting.

This mini project builds **one network that answers the discharge voltage, the capacity and the electrolyte concentration for any rate from 0.5C to 6C**, trained once, and compares it with the DFN on accuracy and on time.

## A2 · The cell, and what is new against Ex10.1

The cell is PyBaMM's `Prada2013` parameter set - the A123 26650 LiFePO4/graphite cell of Prada et al. (J. Electrochem. Soc. 160, 2013), with the open-circuit potentials PyBaMM uses for it (graphite: Chen et al. 2020; LFP: Afshar et al. 2017) - **with both electrode thicknesses multiplied by three**. The tripling turns the thin, high-power A123 into an energy-format LFP cell of 6.9 Ah; it is a device of this project, not a datasheet (**assumed**). It is chosen because it moves the electrolyte limitation into the project's range of rates: with the thin electrodes the SPMe stays within a few millivolts of the DFN even at 10C.

**What is new.** Ex10.1 solved one graphite particle at a constant flux: the **single particle model**, which assumes the electrolyte is perfectly mixed and conducts perfectly, so every particle of an electrode carries the same current. Here the electrolyte is resolved across the cell and coupled to a particle at every place: the full DFN. The cell below prints the data.

In [ ]:
# The cell: three layers, the electrolyte in the pores, a particle at every place
K = mp.constants()
fig, ax = plt.subplots(figsize=(10, 2.2))
mp.shade_regions(ax)                                     # the three layers to scale, in µm
ax.set_yticks([]); ax.set_xlabel("position through the cell x [µm]")
plt.tight_layout(); plt.show()
rows = [("negative electrode", f"graphite, {K['L_n']*1e6:.0f} µm, porosity {K['eps_n']:.2f}, particles R = {K['R_n']*1e6:.1f} µm, D_s = {K['D_n']:.1e} m²/s"),
        ("separator", f"{K['L_s']*1e6:.0f} µm, porosity {K['eps_s']:.2f}"),
        ("positive electrode", f"LiFePO4, {K['L_p']*1e6:.0f} µm, porosity {K['eps_p']:.2f}, particles R = {K['R_p']*1e9:.0f} nm, D_s = {K['D_p']:.1e} m²/s"),
        ("electrolyte", f"{K['c_e0']:.0f} mol/m³ at the start, D_e = {K['D_e']:.1e} m²/s, transference number {K['t_plus']:.2f}"),
        ("electrode area", f"{K['area']:.2f} m² (one electrode pair)"),
        ("capacity", f"{mp.CAP_NOM:.1f} Ah nominal; 1C = {mp.CAP_NOM:.1f} A; cut-off {mp.V_CUT:.1f} V; 25 °C, isothermal")]
for k, v in rows:
    print(f"{k:>20}  {v}")

**What you should see.** The three layers to scale: 102 µm of graphite, 25 µm of separator, 240 µm of LiFePO4. The positive electrode is the thick one, and its particles are a hundred times smaller than the graphite's (50 nm against 10 µm) and diffuse a thousand times slower. Lithium enters the LFP from the electrolyte in the pores, and at high currents the electrolyte near the positive current collector, 370 µm from where most of the lithium is released, runs out first.

## A3 · The model, its equations and its conditions

The DFN is solved through the thickness of the cell, $0 < x < L$ ($x = 0$ the negative current collector, $L$ = 367 µm), and in a spherical particle of radius $R$ at every $x$ of each electrode, $0 < r < R$. It has four unknown fields: the concentration of lithium in the electrolyte $c_e(x, t)$ and in the solid $c_s(x, r, t)$, and the potentials of the electrolyte $\phi_e(x, t)$ and of the solid $\phi_s(x, t)$.

| what | equation | where |
|---|---|---|
| lithium in the electrolyte | $\varepsilon\,\dfrac{\partial c_e}{\partial t} = \dfrac{\partial}{\partial x}\Big(\varepsilon^{1.5} D_e \dfrac{\partial c_e}{\partial x}\Big) + (1 - t^+)\,a\,j$ | the whole cell ($j = 0$ in the separator) |
| lithium in a particle | $\dfrac{\partial c_s}{\partial t} = \dfrac{D_s}{r^2}\dfrac{\partial}{\partial r}\Big(r^2 \dfrac{\partial c_s}{\partial r}\Big)$, $\ -D_s\dfrac{\partial c_s}{\partial r}\Big|_R = j$ | at every $x$ of each electrode: Ex10.1's equation |
| charge in the electrolyte | $\dfrac{\partial}{\partial x}\Big(\kappa_\mathrm{eff}(c_e)\Big[\dfrac{\partial \phi_e}{\partial x} - \dfrac{2(1-t^+)RT}{F}\dfrac{\partial \ln c_e}{\partial x}\Big]\Big) = -F a j$ | the whole cell |
| charge in the solid | $\dfrac{\partial}{\partial x}\Big(\sigma_\mathrm{eff}\dfrac{\partial \phi_s}{\partial x}\Big) = F a j$ | each electrode |
| the reaction | $j = \dfrac{2 j_0}{F}\sinh\Big(\dfrac{F\eta}{2RT}\Big)$, $\ \eta = \phi_s - \phi_e - U(c_s|_R)$, $\ j_0 \propto \sqrt{c_e\, c_s|_R (c_\mathrm{max} - c_s|_R)}$ | each electrode: Butler-Volmer |

$a = 3\varepsilon_s/R$ is the particles' surface per volume of electrode, $U$ the open-circuit potential of the electrode material, $\varepsilon$ the porosity, $t^+$ the transference number, $\kappa_\mathrm{eff}$ and $\sigma_\mathrm{eff}$ the conductivities of the electrolyte and the solid in the porous layer.

| where | condition | why it holds |
|---|---|---|
| both current collectors | $\partial c_e/\partial x = 0$, $\ \partial\phi_e/\partial x = 0$ | lithium ions do not cross the copper or the aluminium |
| the collectors, solid | $-\sigma_\mathrm{eff}\,\partial\phi_s/\partial x = I/A$, $\ \phi_s = 0$ at $x = 0$ | the current enters and leaves through the collectors; the negative one is the reference of potential |
| the electrode-separator faces | $\partial\phi_s/\partial x = 0$ | electrons do not cross the separator |
| the start | $c_e = 1200$ mol/m³ everywhere; $c_s$ uniform in each electrode | the cell has rested, fully charged |
| the end | $V(t) = \phi_s(L) - \phi_s(0) = 2.0$ V | the cut-off: the discharge stops |

The **terminal voltage** $V$ is the output an engineer measures; the **capacity** is the charge passed when $V$ reaches 2.0 V. Every quantity is coupled to every other: the reaction rate depends on both concentrations and both potentials, and changes both concentrations.

## A4 · Why there is no exact solution, and where the SPMe stops

Ex10.1's particle has an exact series because its equation is linear with a constant flux. The DFN is **nonlinear** (the sinh of Butler-Volmer, the square root in $j_0$, the logarithm in the electrolyte's potential, a conductivity that depends on $c_e$) and **coupled** across two length scales: no series or closed form exists, at any rate. The ground truth is therefore numerical (part C).

The SPMe replaces the electrodes by one particle each and linearises the electrolyte about its mean. The cell below compares it with the ground truth at the three test rates.

In [ ]:
# The ground truth (the DFN) and the SPMe at the three test rates
G = np.load("MP10.1A_truth.npz")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for C in mp.C_TEST:
    tag = f"{C:.0f}C"
    l, = ax[0].plot(G["q_" + tag], G["V_" + tag], label=f"DFN, {tag}")
    ax[0].plot(G["spme_q_" + tag], G["spme_V_" + tag], "--", color=l.get_color(), label=f"SPMe, {tag}")
    ax[1].plot(G["x"] * 1e6, G["ce_" + tag][-1], color=l.get_color(), label=f"{tag}, at 95 % of its capacity")
ax[0].set_xlabel("discharged charge q [Ah]"); ax[0].set_ylabel("terminal voltage V [V]"); ax[0].legend(fontsize=7, ncol=2)
mp.shade_regions(ax[1]); ax[1].set_xlabel("x [µm]"); ax[1].set_ylabel("electrolyte concentration c_e [mol/m³]"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for C in mp.C_TEST:
    tag = f"{C:.0f}C"
    print(f"{tag}: capacity, DFN {float(G['cap_' + tag]):.2f} Ah, SPMe {float(G['spme_cap_' + tag]):.2f} Ah; "
          f"lowest electrolyte concentration at 95 % of the capacity {G['ce_' + tag][-1].min():.0f} mol/m³")

**What you should see.** Left, the discharge voltage at 1C, 3C and 5C: the higher the rate, the lower the voltage and the earlier it reaches 2.0 V - 5.81 Ah at 1C, 3.94 Ah at 3C and 2.84 Ah at 5C, against 6.9 Ah nominal. The SPMe (dashed) follows the DFN at 1C, is a few tens of millivolts off at 3C, and **at 5C it stops at 1.39 Ah, half the DFN's capacity**. Right, why: near the end of a 5C discharge the electrolyte in the LiFePO4 next to its current collector has fallen to a few mol/m³ (from 1 200), while at the negative collector it has risen to over 3 000. The SPMe, linearised about the mean concentration, cannot represent an electrolyte that runs out locally; the DFN can.

## A5 · What is asked

| quantity | at | required |
|---|---|---|
| the terminal voltage $V(q)$ against the discharged charge | 1C and 3C, never trained on | within **20 mV** of the ground truth, up to 95 % of its capacity |
| the same | 5C, never trained on | within **50 mV**, up to 95 % of its capacity |
| the capacity to 2.0 V | 5C | within **2 %** |
| the electrolyte concentration $c_e(x)$ | 1C, 3C, 5C, at 25, 50, 75 and 95 % of the capacity | within **30 mol/m³** (2.5 % of 1 200) everywhere |
| the time to answer a new rate online | one Colab runtime | at least **10 times shorter** than the reference model's (part C) |

One network, trained once over 0.5C to 6C. **No training discharge may lie within 0.25C of 1, 3 or 5C** (`mp.simulate` refuses it): the network is scored where it was not fitted.

**The agreed accuracy.** The voltage is scored up to 95 % of the capacity because in the last few per cent it falls by hundreds of millivolts per per cent of charge, and a voltage error there is a capacity error, which is scored on its own. Part C shows that the reference model - the DFN on PyBaMM's default mesh, what an engineer runs - meets the five requirements with a margin of 2 to 18 mV, a 1.17 % capacity error and 15 mol/m³; they are the proposal's numbers, with the window stated.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

The network answers two questions, each a function of the rate:

$$(q,\ C)\ \longmapsto\ V(q; C) \qquad\qquad (x,\ q,\ C)\ \longmapsto\ c_e(x, q; C)$$

the terminal voltage after $q$ ampere-hours discharged at the rate $C$, and the electrolyte concentration at a position $x$ through the cell at the same moment. The capacity at a rate is where its voltage curve reaches 2.0 V. The rate is an **input**: one training covers 0.5C to 6C, and a new rate is answered by evaluating the network.

The discharged charge is used rather than the time because at a constant current the two are proportional ($q = I t$) and the charge puts every rate on one axis: the open-circuit voltage at a given $q$ is the same at every rate.

## B2 · Why not a physics-only PINN of the DFN, and what is built in instead

A PINN of the full DFN would train four coupled fields - $c_e(x, t)$, $\phi_e(x, t)$, $\phi_s(x, t)$ and $c_s(x, r, t)$, the last on two electrodes and four-dimensional with the rate as input - on the five equations of part A, with an exponential (sinh) coupling, a logarithm that diverges where the electrolyte runs out, porosities that jump at the two faces of the separator (so $\partial c_e/\partial x$ jumps there and a smooth network cannot satisfy the equation across them), and a solution that ends at an event, the cut-off, whose instant is itself an unknown. Published physics-only PINNs of the DFN reach tens of millivolts after hours of GPU training at one rate. With a month and a Colab GPU that is not a sound plan, so this project is **hybrid**, as the proposal allows:

* the **physics that has a closed form is built into the network** and holds exactly;
* the **rest is learnt from simulated discharges** of the DFN at rates other than the test rates (the training data, section 3), offline.

| built in | what it gives | how |
|---|---|---|
| the lithium balance of the solids | the mean stoichiometry of each electrode after $q$ Ah, exactly | what leaves the graphite enters the LFP: $\theta_n = \theta_{n,0} - q/Q_n$, $\theta_p = \theta_{p,0} + q/Q_p$ (`mp.stoichiometries`) |
| diffusion in the particles | how far each particle's surface runs ahead of its mean | **Ex10.1's exact series** for a sphere at a constant flux, at its surface (`mp.surface_shift`), with the flux every particle would carry if the reaction were even |
| the open-circuit potentials | the voltage at those surfaces | $U_p(\theta_{p,\mathrm{surf}}) - U_n(\theta_{n,\mathrm{surf}})$ (`mp.ocv_surface`): the single particle model's voltage without kinetics or electrolyte |
| the electrolyte at the collectors | no lithium crosses them | $c_e$ as a cosine series in $x/L$: every $\cos(k\pi x/L)$ has zero slope at both ends |
| conservation of the salt | the lithium in the electrolyte stays what it was | the constant of the cosine series set so that $\int \varepsilon\,c_e\,dx$ is the initial amount |
| the start | $c_e = 1200$ mol/m³ at $q = 0$ | the series multiplied by a function of $q$ that is zero at $q = 0$ |

What the network **learns** is what the DFN adds to that: the overpotentials of the reaction, the potential drop in the electrolyte, and the uneven reaction across the thick electrodes - the very effects that make the SPMe fail at 5C - and the shape of the electrolyte's concentration.

The cell below shows how much of the voltage the built-in physics already carries.

In [ ]:
# The voltage the built-in physics gives, against the ground truth
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for C in mp.C_TEST:
    tag = f"{C:.0f}C"
    q, V = G["q_" + tag], G["V_" + tag]
    l, = ax[0].plot(q, V, label=f"DFN, {tag}")
    ax[0].plot(q, mp.ocv_surface(q, C), ":", color=l.get_color(), label=f"surface open-circuit voltage, {tag}")
    ax[1].plot(q / q[-1] * 100, (mp.ocv_surface(q, C) - V) * 1e3, color=l.get_color(), label=tag)
    rem = mp.ocv_surface(q, C) - V
    print(f"{tag}: what the network must learn, the surface open-circuit voltage minus the DFN's voltage: "
          f"{rem.min()*1e3:.0f} to {rem.max()*1e3:.0f} mV")
ax[0].plot(G["q_1C"], mp.ocv(G["q_1C"]), "k", lw=0.6, label="open-circuit voltage at the mean")
ax[0].set_xlabel("q [Ah]"); ax[0].set_ylabel("V [V]"); ax[0].legend(fontsize=7); ax[0].set_ylim(1.9, 3.7)
ax[1].set_xlabel("share of the capacity [%]"); ax[1].set_ylabel("surface OCV - DFN voltage [mV]"); ax[1].legend()
plt.tight_layout(); plt.show()

**What you should see.** Left, the open-circuit voltage at the mean stoichiometries (black) has the graphite's steps and ends where the graphite is empty; the open-circuit voltage at the particles' surfaces (dotted) shifts those steps and **already carries the knee at the end of each discharge**, at every rate. Right, the remainder - what the network must learn - is smooth: it rises with the rate and slowly through the discharge, 103 to 167 mV at 1C, 209 to 323 mV at 3C and 282 to 492 mV at 5C, with no knee and no steps. A smooth remainder is what a tanh network represents well; the steps and the knee would not be.

## B3 · The training data

`mp.simulate(rates)` solves the DFN at each rate you choose, on a mesh **twice as fine as PyBaMM's default** (40 points in each layer and in each particle's radius): about 0.15 to 0.35 s a discharge on one CPU core, a cost paid once, offline. The finer mesh is the offline cost buying accuracy the online answer keeps: on the default mesh the DFN itself is 18 mV off at 3C (part C), which would leave a network trained on it no room. Each discharge returns its instants $t$ [s], its charge $q$ [Ah], its voltage $V$ [V], its capacity, and the electrolyte concentration at every mesh point $x$ at every instant.

`mp.simulate` refuses any rate within 0.25C of 1, 3 or 5C, and outside 0.5 to 6C. The cell below simulates three rates to show them; part D simulates your training set.

In [ ]:
# Three training discharges: their voltage and their electrolyte at the end
demo = mp.simulate([0.5, 2.0, 6.0])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for d in demo:
    l, = ax[0].plot(d["q"], d["V"], ".", ms=2, label=f"{d['C']:.1f}C, {len(d['q'])} instants")
    ax[1].plot(d["x"] * 1e6, mp.field_at(d, [0.95])[0], color=l.get_color(), label=f"{d['C']:.1f}C, 95 % of its capacity")
ax[0].set_xlabel("q [Ah]"); ax[0].set_ylabel("V [V]"); ax[0].legend(fontsize=8)
mp.shade_regions(ax[1]); ax[1].set_xlabel("x [µm]"); ax[1].set_ylabel("c_e [mol/m³]"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
try:
    mp.simulate([2.9])
except ValueError as e:
    print("refused:", e)

**What you should see.** Three discharges of 300 to 317 instants each, in about a second, closer together where the voltage changes fast (the solver chooses its own steps); the electrolyte concentration at 95 % of each capacity, flat at 0.5C and nearly empty in the positive electrode at 6C. A rate of 2.9C is refused: it is within 0.25C of a test rate.

## B4 · The loss terms

| term | what it compares | scaled by |
|---|---|---|
| $\mathcal{L}_V$ | the network's voltage with the DFN's at every instant of every training discharge | 5 mV |
| $\mathcal{L}_{c_e}$ | the network's electrolyte concentration with the DFN's at the mesh points, at instants spread over each discharge | 10 mol/m³ |

Both are given in part D. The physics of the table in section 2 is built into the network's form, so it needs no loss term. You may add physics as residuals - the electrolyte equation in the separator, where there is no reaction, is the simplest - but read the warning in section 2 first: the porosity jumps at the separator's faces, and a smooth series across them leaves a residual no training removes. We tried it with the cosine series of the solved version: the residual fought the data, and the electrolyte's error grew from about 20 to over 100 mol/m³. A residual needs a representation with a kink at each face (one series per layer, joined by continuity of $c_e$ and of the flux).

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the DFN an engineer runs: PyBaMM on its default mesh | the same DFN on a mesh eight times finer, solver tolerances 10⁻⁹ |
| mesh | 20 points in each layer and in each particle's radius | 160 points in each |
| its job | to be **timed** against the network: the online cost of a new rate | to **score** the network and the reference |

Both are PyBaMM's DFN (`mp.DFN`). The training data of part B is the same model on the mesh twice as fine as the default.

## C2 · The scheme, step by step

1. **Finite volumes in $x$.** The cell's thickness is cut into cells in each layer; the electrolyte's concentration and potential are unknowns at every cell, the solid's potential in the electrodes' cells. Fluxes between cells use the harmonic mean of the effective coefficients, so the flux is continuous where the porosity jumps.
2. **Finite volumes in $r$.** Every cell of an electrode has its own particle, cut into spherical shells; the flux through the outer face is the reaction $j$ of that cell. This is Ex10.1's finite-volume solver, once per cell of the electrode.
3. **Method of lines.** After the discretisation in space the model is a system of differential equations in time (the concentrations) and algebraic equations (the potentials, which have no time derivative): a differential-algebraic system of about 1 300 unknowns on the default mesh and 80 000 on the ground truth's.
4. **Time.** SUNDIALS' IDA solver integrates it with variable-order backward differentiation formulas and adapts its step to the tolerances.
5. **The cut-off as an event.** The solver stops at the instant the terminal voltage reaches 2.0 V, located to the tolerance: that instant times the current is the capacity.
6. **Built once, solved per rate.** The current is an input of the built model, so a new rate costs one solve, not a new discretisation: that is how the reference model is timed.

In [ ]:
# The ground truth: convergence at 5C on meshes each twice as fine, from the truth file
print("mesh   points per layer   capacity [Ah]   V at 50 % [V]   V at 90 % [V]   solve [s]")
for r, n, c, v5, v9, s in zip(G["conv_refine"], G["conv_points"], G["conv_cap"], G["conv_V50"], G["conv_V90"], G["conv_seconds"]):
    print(f" x{int(r)}       {int(n):4d}            {c:.4f}          {v5:.4f}          {v9:.4f}        {s:5.1f}")
dc = np.diff(G["conv_cap"])
print("capacity change between meshes [Ah]:", " ".join(f"{x:+.5f}" for x in dc), "| observed order", np.round(G["conv_order"], 2))
print(f"Richardson extrapolation of the capacity: {float(G['conv_richardson_cap']):.4f} Ah; the ground truth's mesh is "
      f"{abs(G['conv_cap'][-1] / G['conv_richardson_cap'] - 1) * 100:.3f} % from it")
for C in mp.C_TEST:
    e, s, f = G[f"balance_{C:.0f}C"]
    print(f"{C:.0f}C: lithium in the electrolyte kept to {e:.1e}, in the particles to {s:.1e}; "
          f"lithium out of the graphite against the charge passed (Faraday) to {f:.1e}")
for r, z, sp in zip(G["zero_strength_refine"], G["zero_strength_error"], G["zero_strength_spread"]):
    print(f"zero strength, mesh x{int(r)}: against Ex10.1's exact series {z:.1e}, particles' spread {sp:.1e} (of the drop)")

**What you should see.** At 5C the capacity changes by 0.0254, 0.0063 and 0.0016 Ah from mesh to mesh: four times less each time, **second order** (observed 2.01 and 2.00). Richardson extrapolation gives 2.8372 Ah, and the ground truth's mesh is 0.018 % from it - a hundred times finer than the 2 % asked; its voltage changes by 1.2 mV at 90 % of the capacity between the last two meshes, sixteen times finer than the 20 mV asked. The balances: the lithium in the electrolyte and in the particles is kept to rounding ($10^{-13}$ or better), and the lithium leaving the graphite equals the charge passed divided by Faraday's constant to $3\times10^{-6}$ (the solver's tolerance).

**The zero-strength check.** With the electrolyte's diffusivity and conductivity and the solids' conductivities multiplied by 1 000, every particle in the graphite carries the same flux (they differ by $6\times10^{-5}$ of the drop), and the DFN's surface concentration must follow **Ex10.1's exact series**. It does: to $1.5\times10^{-3}$, $3.7\times10^{-4}$ and $9.2\times10^{-5}$ of the drop on meshes x2, x4 and x8 - second order again. The model with the electrolyte switched off is Ex10.1's particle.

## C3 · The reference model: accuracy and time

In [ ]:
# The reference model against the ground truth, and its time per new rate
mp.reference(2.2)                                        # build the model once (not timed: done before any request)
REF = {"V_err": {}, "cap": {}}
for C in mp.C_TEST:
    tag = f"{C:.0f}C"
    q, V, cap = mp.reference(C)
    REF["V_err"][C] = mp.voltage_error(q, V, G["q_" + tag], G["V_" + tag], float(G["cap_" + tag]))
    REF["cap"][C] = cap / float(G["cap_" + tag]) - 1
    d = mp.reference_model().discharge(C)                # the same solve with its electrolyte field
    ce = np.array([np.interp(G["x"], d["x"], row) for row in mp.field_at(d, G["ce_shares"])])
    REF.setdefault("ce_err", {})[C] = np.abs(ce - G["ce_" + tag]).max()
    print(f"{tag}: voltage within {REF['V_err'][C]*1e3:.1f} mV up to 95 % (asked {mp.V_TOL[C]*1e3:.0f}); "
          f"capacity {cap:.3f} Ah, off by {REF['cap'][C]*100:+.2f} %; electrolyte within {REF['ce_err'][C]:.1f} mol/m³")
t_ref = middle(lambda: mp.reference(3.7))                # a new rate: one solve of the built model
REF["t_ref_s"] = t_ref                                   # kept for part E
print(f"\nreference model: {t_ref*1e3:.1f} ms per new rate; the network must answer in under {t_ref*1e3/10:.1f} ms")

**What you should see.** The reference model is within 2.2, 17.8 and 44.2 mV of the ground truth at 1C, 3C and 5C up to 95 % of the capacity, and its 5C capacity is 1.17 % high: inside every requirement of part A, with the least margin at 3C. The electrolyte too: within 1.2, 5.2 and 14.9 mol/m³ at the four scored instants (30 asked). One new rate costs some tens of milliseconds of solving on one CPU core (35 to 78 ms in our runs, depending on what else the machine ran; the time printed above is this runtime's, kept in `REF`). The DFN gains nothing from a GPU at this size, and an engineer would run it on a CPU.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2, including which rates you simulate), and the **figures** (TODO 3).

**The contract.** Your network is a PyTorch module called `net` with two methods:

| method | arguments | returns |
|---|---|---|
| `net.voltage(q, C)` | $q$ the discharged charge [Ah], $C$ the rate [1/h]; `(n, 1)` tensors | the terminal voltage [V], `(n, 1)` |
| `net.electrolyte(x, q, C)` | $x$ the position through the cell [m], 0 to 367 µm; $q$, $C$ as above | the electrolyte concentration [mol/m³], `(n, 1)` |

The capacity at a rate is where `net.voltage` first reaches 2.0 V; `capacity(C)` below finds it. Whatever you compute inside - the built-in physics of part B, features, scaling, several sub-networks - stays inside `net`.

## D2 · The data and the loss, given

`make_data(discharges)` turns a list from `mp.simulate` into tensors: every instant's $(q, C, V)$, and the electrolyte at every mesh point at 40 instants spread evenly over the charge of each discharge. `losses(net, data, n_ce)` returns the two terms of part B, section 4, the electrolyte term on `n_ce` points drawn at random (all of them if `n_ce` is None).

In [ ]:
# The data, the loss and the capacity, given ---------------------------------------------
K = mp.constants()

def T(a):
    """A NumPy array as an (n, 1) tensor on the device."""
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def make_data(discharges, n_instants=40):
    """Tensors of the training discharges: voltage (q, C, V) and electrolyte (x, q, C, c_e)."""
    q = np.concatenate([d["q"] for d in discharges])
    C = np.concatenate([np.full(len(d["q"]), d["C"]) for d in discharges])
    V = np.concatenate([d["V"] for d in discharges])
    xs, qs, Cs, cs = [], [], [], []
    for d in discharges:
        shares = np.linspace(0.0, 1.0, n_instants + 1)[1:]
        F = mp.field_at(d, shares)                       # (instants, mesh points) [mol/m³]
        xs.append(np.tile(d["x"], n_instants)); qs.append(np.repeat(shares * d["cap"], len(d["x"])))
        Cs.append(np.full(F.size, d["C"])); cs.append(F.ravel())
    return dict(q=T(q), C=T(C), V=T(V), x_e=T(np.concatenate(xs)), q_e=T(np.concatenate(qs)),
                C_e=T(np.concatenate(Cs)), c_e=T(np.concatenate(cs)))

def losses(net, data, n_ce=4000, generator=None):
    """The voltage term (every instant, scaled by 5 mV) and the electrolyte term (n_ce random points, 10 mol/m³)."""
    L_V = (((net.voltage(data["q"], data["C"]) - data["V"]) / 0.005) ** 2).mean()
    n = len(data["c_e"])
    i = torch.arange(n, device=DEVICE) if n_ce is None else torch.randint(0, n, (n_ce,), generator=generator, device=DEVICE)
    L_c = (((net.electrolyte(data["x_e"][i], data["q_e"][i], data["C_e"][i]) - data["c_e"][i]) / 10.0) ** 2).mean()
    return {"voltage": L_V, "electrolyte": L_c}

# Ah, where the network's voltage is evaluated: 400 points, dense in the first 5 % of the charge, where the
# voltage falls within seconds of the start, then even up to the nominal capacity
Q_GRID = np.unique(np.r_[0.0, np.geomspace(1e-5, 0.05, 80), np.linspace(0.0, 1.0, 320)]) * mp.CAP_NOM

def curve(C):
    """The network's discharge curve at C on Q_GRID, as a NumPy array [V]."""
    with torch.no_grad():
        return net.voltage(T(Q_GRID), T(np.full(Q_GRID.size, C))).cpu().numpy().ravel()

def capacity(C):
    """The network's capacity at C [Ah]: where its voltage first reaches the 2.0 V cut-off."""
    return mp.crossing(Q_GRID, curve(C))

## D3 · TODO 1 · Your network

Write the class `Net` with the two methods of the contract and make `net = Net().to(DEVICE)`. Decide:

* **what is built in.** Part B, section 2: the functions `mp.ocv_surface(q, C, torch)`, `mp.ocv(q, torch)`, `mp.surface_shift(t, C, electrode, torch)` and `mp.constants()` work on tensors. Using them is the strongest advice this notebook gives; how to use them is yours;
* **what the network learns, and from which inputs.** The rate spans a factor of twelve: $\ln C$ is a better input than $C$. Inputs and outputs of order one;
* **the electrolyte.** A function of $(x, q, C)$; the zero flux at the collectors, the conservation of the salt and the start can be built in (part B, section 2);
* **its size.** Layers, neurons, activation.

In [ ]:
# TODO 1 - your network: net.voltage(q, C) -> V [V] and net.electrolyte(x, q, C) -> c_e [mol/m³], (n, 1) tensors
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers

    def voltage(self, q, C):
        ...                                              # return V, shape (n, 1)

    def electrolyte(self, x, q, C):
        ...                                              # return c_e, shape (n, 1)

torch.manual_seed(101)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Simulate your training set with `mp.simulate(rates)` (which rates, and how many, is yours - not within 0.25C of a test rate), turn it into tensors with `make_data`, and train `net` with `losses`. Decide the optimiser or optimisers, the steps, the learning rate and its schedule, and the weight of each term. Keep the history of each term for TODO 3, and **measure the offline time**, simulation included: set `TRAIN_SECONDS`.

`quick_score()` below gives in a second the worst voltage error at the three test rates [mV], the 5C capacity error [%] and the worst electrolyte error [mol/m³].

In [ ]:
# A quick score, given: the worst errors at the test rates, against the ground truth
G = np.load("MP10.1A_truth.npz")

def electrolyte_errors(C):
    """The largest |c_e error| [mol/m³] at each scored share of the capacity, at the same discharged charge."""
    tag = f"{C:.0f}C"; cap = float(G["cap_" + tag]); x = G["x"]
    out = []
    for i, s in enumerate(G["ce_shares"]):
        with torch.no_grad():
            c = net.electrolyte(T(x), T(np.full(x.size, s * cap)), T(np.full(x.size, C))).cpu().numpy().ravel()
        out.append(np.abs(c - G["ce_" + tag][i]).max())
    return np.array(out)

def quick_score():
    v = max(mp.voltage_error(Q_GRID, curve(C), G[f"q_{C:.0f}C"], G[f"V_{C:.0f}C"], float(G[f"cap_{C:.0f}C"])) for C in mp.C_TEST)
    c5 = capacity(5.0) / float(G["cap_5C"]) - 1
    e = max(electrolyte_errors(C).max() for C in mp.C_TEST)
    return v * 1e3, c5 * 100, e

In [ ]:
# TODO 2 - your training
history = {"voltage": [], "electrolyte": []}
t0 = time.perf_counter()
...                                                      # simulate your rates, make_data, then train; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"offline {TRAIN_SECONDS:.0f} s; worst voltage error %.1f mV, 5C capacity %+.2f %%, electrolyte %.1f mol/m³" % quick_score())

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the voltage at one test rate against the ground truth with its error. Save each figure you want in the report as `MP10.1A_report_<name>.png` (in `cc.output_path`).

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores and keeps the answers in the dictionary `R`:

* the discharge curve on `Q_GRID` and the capacity at the test rates 1C, 3C and 5C, and at 0.5 to 6C in steps of 0.25C (the capacity against the rate);
* the electrolyte concentration at the test rates at the scored shares of the capacity, on the ground truth's points;
* the **online time** of a new rate on this runtime - the discharge curve and the capacity, from one call of `net.voltage` - and the same for 10 and 100 rates at once, against the reference model run 1, 10 and 100 times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()

def answer(rates):
    """The network's discharge curves on Q_GRID [V] and capacities [Ah] for several rates, from one call of net.voltage."""
    rates = np.atleast_1d(rates)
    with torch.no_grad():
        V = net.voltage(T(np.tile(Q_GRID, len(rates))), T(np.repeat(rates, Q_GRID.size))).reshape(len(rates), -1).cpu().numpy()
    return V, np.array([mp.crossing(Q_GRID, v) for v in V])

out = {"q": Q_GRID, "rates_test": np.array(mp.C_TEST), "rates_sweep": np.arange(0.5, 6.01, 0.25)}
out["V_test"], out["cap_test"] = answer(out["rates_test"])
out["V_sweep"], out["cap_sweep"] = answer(out["rates_sweep"])
for C in mp.C_TEST:
    tag = f"{C:.0f}C"; cap = float(G["cap_" + tag]); x = G["x"]
    with torch.no_grad():
        out["ce_" + tag] = np.array([net.electrolyte(T(x), T(np.full(x.size, s * cap)), T(np.full(x.size, C))).cpu().numpy().ravel()
                                     for s in G["ce_shares"]])
# the reference model's capacity over the same sweep, for comparison (about 3 s)
out["ref_cap_sweep"] = np.array([mp.reference(C)[2] for C in out["rates_sweep"]])

timing = {}
for k in (1, 10, 100):
    rates = np.linspace(0.6, 5.9, k)
    timing[f"net_{k}"] = middle(lambda: answer(rates))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(C) for C in rates], reps=3 if k == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP10.1A_net.pt"))   # the trained weights, beside the report
print(f"online, one new rate: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.1f} ms "
      f"({timing['ref_1']/timing['net_1']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth `G` and the reference model `REF` of part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the ten questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP10.1A_truth.npz")
print("in R:", ", ".join(R))
print(f"reference model (part C): {REF['t_ref_s']*1e3:.1f} ms per new rate; voltage errors "
      + ", ".join(f"{C:.0f}C {v*1e3:.1f} mV" for C, v in REF["V_err"].items()))

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* the largest voltage error up to 95 % of the capacity at 1C, 3C and 5C [mV] (`mp.voltage_error`), the network's and the reference model's;
* the capacity error at 1C, 3C and 5C [%], the network's and the reference model's;
* the largest electrolyte error at each test rate [mol/m³];
* the network's online time for one new rate, the reference model's (both in `R`), and their ratio;
* the offline time.

Then say in one line each whether the requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>50}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP10.1A_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: the voltage at the three test rates against the ground truth, with the error; the electrolyte concentration at 5C at the four scored shares against the ground truth; the capacity against the rate (`R["cap_sweep"]`, `R["ref_cap_sweep"]`); and the online times for 1, 10 and 100 rates (`R["net_1"]`, `R["ref_10"]` and so on).

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the ten questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 5 are about your network and its result; 6 to 9 ask you to use it; 10 concludes. The next cell writes `MP10.1A_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your network: what is built in (which physics, and how), what is learnt, its inputs, layers, neurons and number of weights. Why this design?": """
""",
    "2. Your training data and training: which rates you simulated and why, the optimiser(s), steps, weights of the terms, and the offline time in seconds (simulation included).": """
""",
    "3. The voltage at 1C, 3C and 5C: give the network's largest error up to 95 % of the capacity [mV] against the reference model's. Which requirements are met, and where in the discharge is the error largest?": """
""",
    "4. The capacity at 1C, 3C and 5C [%], the network's and the reference model's errors. Why is the capacity accurate even where the voltage falls by hundreds of millivolts per per cent of charge?": """
""",
    "5. The online time for one new rate: the network against the reference model, in milliseconds, and the ratio. Then for 10 and 100 rates: how do the two grow?": """
""",
    "6. The electrolyte at 5C: where does it run out, at what share of the capacity, and how large is the network's error there [mol/m³]? Relate it to the SPMe's capacity at 5C.": """
""",
    "7. How much of the voltage does the built-in physics carry? Compare the surface open-circuit voltage with the DFN at 1C and 5C, and say what the network had to learn (in mV).": """
""",
    "8. The capacity against the rate from 0.5C to 6C (your sweep against the reference model's): where do they differ most, and what does that say about interpolation between the training rates?": """
""",
    "9. Train once more with fewer rates (for example five), or without the built-in surface open-circuit voltage: give the voltage error at 3C and 5C and say what it shows.": """
""",
    "10. To conclude: a pack designer asks for the capacity at 200 load profiles' rates. Which method would you give them, given your measured accuracy, online and offline times? What would change for a cell with other parameters?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP10.1A — The Full Model, Where the SPMe Stops", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP10.1A_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP10.1A_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP10.1A_report.md into MP10.1A_report.pdf, with any figure
# saved as MP10.1A_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP10.1A_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP10.1A_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP10.1A_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP10.1A_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass